In [ ]:
import os
import sys

import torch
from ultralytics import RTDETR


# Add project root to system path (for relative imports to work)
project_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src import config
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
torch.cuda.empty_cache()

Torch: 2.8.0+cu128
CUDA available: False


In [3]:
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

weights_path = f"{config.WORKSPACE_ROOT}/data/runs/rtdetr/train/motor_rtdetr_l_1024/weights/best.pt"

model = RTDETR(weights_path)

image_glob = "/data/horse/ws/kein254g-team_project/test_flat/**/*.jpg"
save_dir_project = "/data/horse/ws/kein254g-team_project/rtdetr_batch"
save_dir_name = "exp"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

predict_args = dict(
    source=image_glob,
    imgsz=512,  # lower resolution
    conf=0.60,  # fewer candidates -> faster NMS
    device="cpu",  # explicit
    half=False,  # IMPORTANT on CPU
    batch=8,  # increase if RAM allows
    workers=2,  # tune: 0–4 on HPC shared FS
    stream=True,
    save=True,  # cut I/O
    save_txt=True,
    save_conf=True,
    project=save_dir_project,
    name=save_dir_name,
    max_det=50,  # reduce NMS work
)

try:
    results = model.predict(**predict_args)

except RuntimeError as e:
    msg = str(e)
    print("RuntimeError:", msg)
    if "CUDA out of memory" in msg or "CUDNN_STATUS_ALLOC_FAILED" in msg:
        print("\nOOM detected. Retrying with smaller settings...")
        torch.cuda.empty_cache()
        predict_args.update(dict(imgsz=768, batch=1, half=True))
        results = model.predict(**predict_args)
    else:
        raise

Torch: 2.8.0+cu128
CUDA available: False
